In [53]:
import os
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, ConcatDataset, random_split
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, f1_score
import numpy as np
from PIL import Image

In [54]:
train_normal=len(os.listdir('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train/NORMAL'))
train_pneumonia=len(os.listdir('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train/PNEUMONIA'))
print(f'Normal: {train_normal}, Pnuemonia: {train_pneumonia}')

Normal: 1341, Pnuemonia: 3875


In [55]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [56]:
# Increasing validation instances
train_raw = datasets.ImageFolder('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train')
val_raw = datasets.ImageFolder('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/val')
full_train = ConcatDataset([train_raw, val_raw])
train_size=int(0.8*len(full_train))
val_size=len(full_train)-train_size
train_subset, val_subset = random_split(full_train, [train_size, val_size])
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, idx):
        x, y = self.subset[idx]
        return self.transform(x), y
    def __len__(self):
        return len(self.subset)
        
train_dataset = TransformSubset(train_subset, train_transform)
val_dataset = TransformSubset(val_subset, val_transform)
test_dataset = datasets.ImageFolder('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test',
                                   transform=val_transform)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader= DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader= DataLoader(test_dataset, batch_size=32, shuffle=False)

In [57]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels.unique())

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([0, 1])


In [58]:
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

for p in model.parameters():
    p.requires_grad=False

num_features=model.classifier.in_features

model.classifier=nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)

In [59]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total: {total:,} | Trainable: {trainable:,}')

Total: 7,216,770 | Trainable: 262,914


In [60]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=model.to(device)
class_weights=torch.tensor([3.0, 1.0]).to(device)
criterion=nn.CrossEntropyLoss(weight=class_weights)
optimizer=torch.optim.Adam(model.parameters(), lr=0.0001)

In [61]:
for epoch in range(10):
    model.train()
    train_loss=0.0
    correct_train=0
    total_train=0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss+=loss.item()*images.size(0)

        _, predictions=torch.max(outputs, 1)
        correct_train+=(predictions==labels).sum().item()
        total_train+=labels.size(0)

    epoch_train_loss=train_loss/len(train_loader.dataset)
    epoch_train_acc=correct_train/total_train

    model.eval()
    val_loss=0.0
    correct_val=0
    total_val=0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs=model(images)
            loss=criterion(outputs, labels)
         
            val_loss+=loss.item()*images.size(0)
            _, predictions=torch.max(outputs, 1)
            correct_val+=(predictions==labels).sum().item()
            total_val+=labels.size(0)
        epoch_val_loss=val_loss/len(val_loader.dataset)
        epoch_val_acc=correct_val/total_val

    print(f'Epoch {epoch+1}/10 | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}')

Epoch 1/10 | Train Loss: 0.4807 | Train Acc: 0.8280 | Val Loss: 0.3126 | Val Acc: 0.8787
Epoch 2/10 | Train Loss: 0.2810 | Train Acc: 0.9013 | Val Loss: 0.2489 | Val Acc: 0.8634
Epoch 3/10 | Train Loss: 0.2180 | Train Acc: 0.9195 | Val Loss: 0.1977 | Val Acc: 0.9035
Epoch 4/10 | Train Loss: 0.1975 | Train Acc: 0.9211 | Val Loss: 0.1835 | Val Acc: 0.9083
Epoch 5/10 | Train Loss: 0.1897 | Train Acc: 0.9293 | Val Loss: 0.1736 | Val Acc: 0.9179
Epoch 6/10 | Train Loss: 0.1893 | Train Acc: 0.9264 | Val Loss: 0.1667 | Val Acc: 0.9207
Epoch 7/10 | Train Loss: 0.1695 | Train Acc: 0.9395 | Val Loss: 0.1545 | Val Acc: 0.9341
Epoch 8/10 | Train Loss: 0.1740 | Train Acc: 0.9360 | Val Loss: 0.1500 | Val Acc: 0.9341
Epoch 9/10 | Train Loss: 0.1597 | Train Acc: 0.9400 | Val Loss: 0.1660 | Val Acc: 0.9226
Epoch 10/10 | Train Loss: 0.1745 | Train Acc: 0.9395 | Val Loss: 0.1590 | Val Acc: 0.9265


In [62]:
model.eval()
test_loss=0.0
correct_test=0
total_test=0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs=model(images)
        loss=criterion(outputs, labels)
        test_loss+=loss.item()*images.size(0)
        _, predictions = torch.max(outputs, 1)
        correct_test+=(predictions==labels).sum().item()
        total_test+=labels.size(0)

print(f'Test Loss: {test_loss/len(test_loader.dataset):.4f} | Test Acc: {correct_test/total_test:.4f}')

Test Loss: 0.3215 | Test Acc: 0.8750


In [63]:
all_preds = []
all_labels = []
all_probs = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]  # probability of Pneumonia class
        _, predictions = torch.max(outputs, 1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=['Normal', 'Pneumonia']))
print(f'AUC-ROC: {roc_auc_score(all_labels, all_probs):.4f}')

[[187  47]
 [ 31 359]]
              precision    recall  f1-score   support

      Normal       0.86      0.80      0.83       234
   Pneumonia       0.88      0.92      0.90       390

    accuracy                           0.88       624
   macro avg       0.87      0.86      0.86       624
weighted avg       0.87      0.88      0.87       624

AUC-ROC: 0.9403


In [ ]:
# Test accuracy: 88%
# Normal recall: 80%
# Pneumonia recall: 92%
# Macro F1: 0.86
# AUC-ROC: 0.9403